# Test modeling: T11–T15, T25 và T31–T37

Các mục được trình bày theo thứ tự: **T11 → T12 → T13 → T14 → T15 → T25 → T31 → T32 → T36 → T37**.

Chạy notebook từ trên xuống bằng **Run All**. Phần khởi tạo thực hiện split T13 trước mọi bước fit; các yêu cầu được mô tả và kiểm chứng trong các mục tương ứng. Notebook chính `02_Modeling_Evaluation.ipynb` được giữ nguyên.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import cloudpickle
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_SEED = 42
TEST_SIZE = 0.20
MISSING_RATE = 0.05

cwd = Path.cwd()
notebook_output_candidates = [cwd, cwd / "notebooks", cwd / "03_SourceCode" / "notebooks"]
notebook_output_dir = next(
    (path.resolve() for path in notebook_output_candidates
     if path.name.lower() == "notebooks" and path.is_dir()),
    cwd.resolve(),
)
data_candidates = [
    cwd / "../../data/insurance.csv",
    cwd / "../data/insurance.csv",
    cwd / "data/insurance.csv",
]
data_path = next((path.resolve() for path in data_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Không tìm thấy data/insurance.csv. Hãy chạy từ thư mục dự án hoặc notebooks.")

df = pd.read_csv(data_path)
X = df.drop(columns="charges")
y = df["charges"]

# T13 prerequisite: split raw rows before any preprocessing or model fitting.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=X["smoker"],
)
print(f"Dữ liệu: {df.shape}; đường dẫn: {data_path}")
print("Missing trong feature gốc:")
print(X.isna().sum())
print("Tổng số ô missing:", int(X.isna().sum().sum()))

## T11 – Chiến lược dữ liệu thiếu (Imputer trong Pipeline)

### Yêu cầu

1. Kiểm tra và báo cáo missing trong dữ liệu gốc (hiện dữ liệu gần như không thiếu).
2. Vẫn dùng `SimpleImputer` trong Pipeline để xử lý dữ liệu đầu vào phát sinh missing và giữ quy trình triển khai nhất quán.
3. Biến số dùng median; biến phân loại dùng most frequent.
4. Imputer chỉ fit trên training fold trong CV hoặc trên train khi fit mô hình cuối.
5. Mô phỏng missing trên train để kiểm tra độ bền; không sửa dữ liệu gốc và không tác động test.

Train/test đã được chia ở phần khởi tạo trước mọi bước fit. Cell tiếp theo tạo bản sao train có 5% missing ngẫu nhiên; kết quả đánh giá độ bền nằm trong phần T25.

In [ ]:
def simulate_missing(data, rate=MISSING_RATE, seed=RANDOM_SEED):
    simulated = data.copy()
    rng = np.random.default_rng(seed)
    for column in simulated.columns:
        mask = rng.random(len(simulated)) < rate
        simulated.loc[mask, column] = np.nan
    return simulated

X_train_missing = simulate_missing(X_train, seed=RANDOM_SEED)
print("Missing giả lập theo cột trong train:")
print(X_train_missing.isna().sum())
print("Tổng ô missing giả lập:", int(X_train_missing.isna().sum().sum()))

## T12 – Kỹ nghệ đặc trưng (Feature Engineering)

### Yêu cầu

Tạo đủ 5 feature chỉ từ đầu vào, không dùng `charges`:

| Feature | Quy tắc |
| --- | --- |
| `bmi_category` | Nhóm BMI WHO: <18.5; 18.5–<25; 25–<30; 30–<35; 35–<40; ≥40 |
| `age_group` | `<30`, `30–44`, `45–59`, `≥60` (nhóm thực nghiệm, không phải ngưỡng y khoa) |
| `smoker_bmi` | BMI × 1 nếu smoker=yes; BMI × 0 nếu smoker=no |
| `obese_smoker` | 1 nếu BMI ≥30 và smoker=yes; ngược lại 0 |
| `age_squared` | `age ** 2` |

Giữ feature tương tác là missing nếu một đầu vào cần thiết bị thiếu; Imputer trong Pipeline sẽ xử lý. Cell dưới đây kiểm tra quy tắc và ngưỡng; mức hữu ích từng feature được đánh giá bằng ablation ở T25.

In [ ]:
NUMERIC_INPUTS = ["age", "bmi", "children"]
BINARY_INPUTS = ["sex", "smoker"]
NOMINAL_INPUTS = ["region"]
ENGINEERED_FEATURES = {
    "bmi_category", "age_group", "smoker_bmi", "obese_smoker", "age_squared"
}

def engineer_features(data, enabled_features):
    result = data.copy()
    if "bmi_category" in enabled_features:
        result["bmi_category"] = pd.cut(
            result["bmi"],
            bins=[-np.inf, 18.5, 25, 30, 35, 40, np.inf],
            labels=["underweight", "normal", "overweight", "obesity_I", "obesity_II", "obesity_III"],
            right=False,
        ).astype(object)
    if "age_group" in enabled_features:
        result["age_group"] = pd.cut(
            result["age"],
            bins=[-np.inf, 30, 45, 60, np.inf],
            labels=["under_30", "30_44", "45_59", "60_plus"],
            right=False,
        ).astype(object)

    smoker_indicator = result["smoker"].map({"no": 0.0, "yes": 1.0})
    if "smoker_bmi" in enabled_features:
        result["smoker_bmi"] = result["bmi"] * smoker_indicator
    if "obese_smoker" in enabled_features:
        is_missing = result["bmi"].isna() | smoker_indicator.isna()
        result["obese_smoker"] = (
            ((result["bmi"] >= 30) & (smoker_indicator == 1))
            .astype(float)
            .mask(is_missing)
        )
    if "age_squared" in enabled_features:
        result["age_squared"] = result["age"] ** 2
    return result

engineered_preview = engineer_features(X_train_missing.head(8), ENGINEERED_FEATURES)
required_features = sorted(ENGINEERED_FEATURES)
assert set(required_features).issubset(engineered_preview.columns)
display(engineered_preview[required_features].head())

boundary_probe = pd.DataFrame({
    "age": [29, 30, 45, 60],
    "sex": ["female", "male", "female", "male"],
    "bmi": [18.49, 18.5, 30, 40],
    "children": [0, 0, 0, 0],
    "smoker": ["yes", "no", "yes", "yes"],
    "region": ["northwest"] * 4,
})
boundary_features = engineer_features(boundary_probe, ENGINEERED_FEATURES)
assert boundary_features["bmi_category"].tolist() == [
    "underweight", "normal", "obesity_I", "obesity_III"
]
assert boundary_features["age_group"].tolist() == [
    "under_30", "30_44", "45_59", "60_plus"
]
assert boundary_features["smoker_bmi"].tolist() == [18.49, 0.0, 30.0, 40.0]
assert boundary_features["obese_smoker"].tolist() == [0.0, 0.0, 1.0, 1.0]
print("Các ngưỡng BMI, tuổi và feature tương tác đã qua kiểm tra.")

### T12.1 – Đánh giá mức hữu ích từng feature

So sánh baseline chỉ có feature gốc, mô hình có đủ 5 feature mới và các mô hình lần lượt bỏ một feature. Dùng cùng folds trong cross-validation.

`RMSE_increase_vs_full > 0` nghĩa là bỏ feature làm RMSE tăng trong thí nghiệm; giá trị âm nghĩa là RMSE giảm. Đối chiếu chênh lệch với độ lệch chuẩn giữa folds: chênh lệch nhỏ so với độ lệch chuẩn chưa phải bằng chứng feature thực sự hữu ích.

## T13 – Chia Train/Test trước mọi bước fit

### Yêu cầu

- Tỉ lệ train/test 80/20, `random_state` cố định.
- Phân tầng theo `smoker` để giữ tỉ lệ nhóm hút thuốc tương tự.
- Split được chạy ngay trong cell khởi tạo dữ liệu, trước T11/T12 và trước mọi fit.
- Test holdout không được dùng cho fit, CV, chọn feature hoặc chọn mô hình; chỉ được dùng ở T31–T32 sau khi khóa lựa chọn bằng CV.

Cell khởi tạo đã thực hiện split. Cell tiếp theo chỉ báo cáo kích thước train; không đọc nội dung hay tính metric trên test.

In [ ]:
print(f"Train: {len(X_train)} dòng ({len(X_train) / len(X):.0%})")
print("Test holdout đã được dành riêng; chỉ đánh giá ở T31–T32.")

## T14 – Xây dựng `ColumnTransformer`

### Yêu cầu

- **Biến số:** `SimpleImputer(median)` → `StandardScaler`.
- **Biến nhị phân:** điền missing bằng giá trị phổ biến nhất, rồi ánh xạ `female=0`, `male=1`; `no=0`, `yes=1`.
- **Biến phân loại danh nghĩa:** điền missing bằng giá trị phổ biến nhất, rồi dùng `OneHotEncoder(handle_unknown="ignore")`.
- Đặt feature engineering, tiền xử lý và estimator trong cùng Pipeline để mỗi CV fold tự fit các bước trên phần train của fold.

`sex` và `smoker` được ánh xạ nhị phân; `region`, `bmi_category`, `age_group` dùng One-Hot.

In [ ]:
def build_pipeline(enabled_features=None, estimator=None):
    if enabled_features is None:
        enabled_features = set()
    if estimator is None:
        estimator = LinearRegression()

    numeric_features = NUMERIC_INPUTS.copy()
    binary_features = BINARY_INPUTS.copy()
    nominal_features = NOMINAL_INPUTS.copy()
    for feature in ["smoker_bmi", "obese_smoker", "age_squared"]:
        if feature in enabled_features:
            numeric_features.append(feature)
    for feature in ["bmi_category", "age_group"]:
        if feature in enabled_features:
            nominal_features.append(feature)

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    binary_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("mapping", OrdinalEncoder(
            categories=[["female", "male"], ["no", "yes"]],
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])
    nominal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("binary", binary_pipeline, binary_features),
        ("nominal", nominal_pipeline, nominal_features),
    ])
    return Pipeline([
        ("feature_engineering", FunctionTransformer(
            engineer_features,
            kw_args={"enabled_features": enabled_features},
            validate=False,
        )),
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])

print("Đã cấu hình imputer, scaler, binary mapping và one-hot encoding trong Pipeline.")

## T15 – Khai báo mô hình baseline

### Yêu cầu

- So sánh `DummyRegressor(strategy="mean")` với `LinearRegression`.
- Bao gồm mô hình feature gốc, mô hình đủ feature mới và các biến thể ablation T12.
- Các candidate được khai báo trước khi chạy CV; lựa chọn dùng RMSE CV, không xem test.

Cell tiếp theo chỉ khai báo Pipeline. Cross-validation và lựa chọn mô hình được thực hiện bằng hàm chung ở T25.

In [ ]:
candidate_pipelines = {
    "DummyRegressor (mean)": build_pipeline(
        set(), DummyRegressor(strategy="mean")
    ),
    "LinearRegression (raw features)": build_pipeline(set()),
    "LinearRegression (all engineered)": build_pipeline(ENGINEERED_FEATURES),
}
for feature in sorted(ENGINEERED_FEATURES):
    candidate_pipelines[f"LinearRegression (without {feature})"] = build_pipeline(
        ENGINEERED_FEATURES - {feature}
    )
print("Số candidate đã khai báo trước CV:", len(candidate_pipelines))

## T25 – Khung đánh giá chung: K-Fold CV và bộ chỉ số

### Yêu cầu

- Hàm dùng chung gọi `cross_validate` với `KFold(shuffle=True)`; hỗ trợ `k=5` hoặc `k=10`.
- Báo cáo MAE, RMSE, R², MAPE (%) theo mean ± std giữa folds.
- Dùng RMSE làm tiêu chí xếp hạng; MAPE được chọn thay RMSLE vì mô hình tuyến tính có thể dự đoán âm. MAPE nhạy với target nhỏ nên là metric bổ sung.
- Dùng train cho T11 robustness, T12 ablation và T15 model comparison; lưu bảng CV.
- Test holdout tuyệt đối không truyền vào CV.

In [ ]:
def evaluate_models_cv(models, features, target, n_splits=5, include_mape=True):
    if n_splits not in (5, 10):
        raise ValueError("n_splits phải là 5 hoặc 10.")

    scoring = {
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2",
    }
    if include_mape:
        scoring["MAPE"] = "neg_mean_absolute_percentage_error"

    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    rows = []
    for model_name, pipeline in models.items():
        scores = cross_validate(
            pipeline,
            features,
            target,
            cv=splitter,
            scoring=scoring,
            return_train_score=False,
            error_score="raise",
        )
        row = {"model": model_name, "folds": n_splits}
        for metric in ["MAE", "RMSE"]:
            values = -scores[f"test_{metric}"]
            row[metric] = values.mean()
            row[f"{metric}_std"] = values.std(ddof=1)
        r2_values = scores["test_R2"]
        row["R2"] = r2_values.mean()
        row["R2_std"] = r2_values.std(ddof=1)
        if include_mape:
            mape_values = -scores["test_MAPE"] * 100
            row["MAPE_%"] = mape_values.mean()
            row["MAPE_std_%"] = mape_values.std(ddof=1)
        rows.append(row)

    return pd.DataFrame(rows).set_index("model").sort_values("RMSE")

### T25.1 – Chạy đánh giá T11, T12 và T15

Cùng hàm CV được dùng cho: (1) kiểm tra Imputer trên train sạch và train có 5% missing; (2) ablation từng feature; (3) xếp hạng candidate T15. Lựa chọn được khóa theo mean RMSE CV trước khi đi tới test.

In [ ]:
t11_clean_cv = evaluate_models_cv(
    {"T11 - train sạch": build_pipeline(set())}, X_train, y_train, n_splits=5
)
t11_missing_cv = evaluate_models_cv(
    {"T11 - train missing giả lập 5%": build_pipeline(set())},
    X_train_missing,
    y_train,
    n_splits=5,
)
t11_robustness_results = pd.concat([t11_clean_cv, t11_missing_cv])

# Main modeling and feature selection use the original, clean training data.
t12_models = {
    "Baseline - raw features": build_pipeline(set()),
    "Full - all engineered features": build_pipeline(ENGINEERED_FEATURES),
}
for feature in sorted(ENGINEERED_FEATURES):
    t12_models[f"Without {feature}"] = build_pipeline(ENGINEERED_FEATURES - {feature})
t12_cv_results = evaluate_models_cv(t12_models, X_train, y_train, n_splits=5)
full_rmse = t12_cv_results.loc["Full - all engineered features", "RMSE"]
ablation_results = t12_cv_results.loc[
    [f"Without {feature}" for feature in sorted(ENGINEERED_FEATURES)]
].copy()
ablation_results["RMSE_increase_vs_full"] = ablation_results["RMSE"] - full_rmse
ablation_results = ablation_results.sort_values("RMSE_increase_vs_full", ascending=False)

candidate_results = evaluate_models_cv(
    candidate_pipelines, X_train, y_train, n_splits=5
)
best_model_name = candidate_results.index[0]

print("T11 - độ bền Imputer; train-only CV:")
display(t11_robustness_results.round(3))
print("T12 - ablation feature; train-only CV:")
display(ablation_results.round(3))
print("T15 - model comparison; train-only CV:")
display(candidate_results.round(3))
print("Đã khóa candidate theo mean RMSE CV:", best_model_name)

all_cv_results = pd.concat([
    t11_robustness_results.assign(experiment="T11 imputer robustness"),
    t12_cv_results.assign(experiment="T12 feature ablation"),
    candidate_results.assign(experiment="T15 baseline comparison"),
]).reset_index()
cv_results_path = notebook_output_dir / "t25_cv_results.csv"
all_cv_results.to_csv(cv_results_path, index=False)
print("Đã lưu bảng CV:", cv_results_path)

## T31 – Đánh giá cuối trên Test Set độc lập

Mô hình đã được khóa theo RMSE CV ở T25. Tại đây mới đánh giá test holdout cho các candidate đã khai báo trước theo yêu cầu T32; toàn bộ kết quả chỉ dùng để báo cáo, **không tune hoặc đổi mô hình theo test**.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE_%": np.mean(np.abs((y_true - y_pred) / y_true)) * 100,
    }

test_rows = []
for model_name, pipeline in candidate_pipelines.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    assert np.isfinite(predictions).all()
    test_rows.append({"model": model_name, **regression_metrics(y_test, predictions)})

test_results = pd.DataFrame(test_rows).set_index("model")


## T32 – Bảng so sánh đa chỉ số và biểu đồ

Ghép CV mean ± std với Test MAE, RMSE, R², MAPE cho từng candidate. Biểu đồ đối chiếu CV RMSE (kèm std) và Test RMSE. Cột `Selected_by_CV` đánh dấu lựa chọn đã khóa trước khi đọc kết quả test.

In [ ]:
comparison = candidate_results.add_prefix("CV_").join(test_results.add_prefix("Test_"))
comparison["Selected_by_CV"] = comparison.index == best_model_name

comparison_report = pd.DataFrame(index=comparison.index)
for metric in ["MAE", "RMSE", "R2"]:
    comparison_report[f"CV {metric} mean ± std"] = comparison.apply(
        lambda row: f"{row[f'CV_{metric}']:.3f} ± {row[f'CV_{metric}_std']:.3f}", axis=1
    )
comparison_report["CV MAPE mean ± std (%)"] = comparison.apply(
    lambda row: f"{row['CV_MAPE_%']:.2f} ± {row['CV_MAPE_std_%']:.2f}", axis=1
)
for metric in ["MAE", "RMSE", "R2", "MAPE_%"]:
    comparison_report[f"Test {metric}"] = comparison[f"Test_{metric}"]
comparison_report["Chọn theo CV"] = comparison["Selected_by_CV"]

print("T32 - bảng CV mean ± std và kết quả Test cho tất cả mô hình:")
display(comparison_report.round(3))

plot_data = comparison.sort_values("CV_RMSE")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(plot_data.index, plot_data["CV_RMSE"], xerr=plot_data["CV_RMSE_std"])
axes[0].set_title("Cross-validation RMSE (mean ± std)")
axes[0].set_xlabel("RMSE")
axes[1].barh(plot_data.index, plot_data["Test_RMSE"])
axes[1].set_title("Holdout Test RMSE (report only)")
axes[1].set_xlabel("RMSE")
plt.tight_layout()
display(fig)
plt.close(fig)

comparison_path = notebook_output_dir / "t31_t32_model_comparison.csv"
comparison.reset_index().rename(columns={"index": "model"}).to_csv(comparison_path, index=False)
print("Đã lưu bảng so sánh:", comparison_path)
print("Mô hình được chọn theo CV (không đổi theo Test):", best_model_name)

## T36 – Tái lập quy trình mô hình hóa trong notebook kiểm thử

- Chạy toàn bộ cell từ trên xuống trong kernel mới; không dựa vào biến của lần chạy trước.
- Cố định `random_state=42` cho train/test split, missing mô phỏng và K-Fold CV.
- Giữ các chú thích phân biệt CV, test holdout và bước huấn luyện mô hình triển khai.
- Kết quả CV và holdout được ghi ra CSV để đối chiếu giữa các lần chạy.


## T37 – Đóng gói mô hình tối ưu

Sau khi báo cáo đánh giá holdout và khóa lựa chọn bằng CV, fit lại candidate đã chọn trên toàn bộ dữ liệu có nhãn để tạo artifact triển khai. Lưu **toàn bộ Pipeline** (feature engineering, imputer, scaler, encoder và estimator), không chỉ riêng estimator. Artifact được lưu tại `03_SourceCode/best_model.joblib` và nạp lại để kiểm tra dự đoán mẫu.


In [ ]:
# Refit after holdout reporting; test results do not affect model selection.
final_model_pipeline = candidate_pipelines[best_model_name]
final_model_pipeline.fit(X, y)

model_output_path = notebook_output_dir.parent / "best_model.joblib"
with model_output_path.open("wb") as model_file:
    cloudpickle.dump(final_model_pipeline, model_file)

# Reload the artifact to verify preprocessing and prediction are serialized together.
loaded_model_pipeline = joblib.load(model_output_path)
assert {"feature_engineering", "preprocessor", "model"}.issubset(
    loaded_model_pipeline.named_steps
)
packaging_predictions = loaded_model_pipeline.predict(X.iloc[:5])
assert np.isfinite(packaging_predictions).all()

print("Candidate được đóng gói:", best_model_name)
print("Pipeline đã lưu tại:", model_output_path)
print("Đã nạp lại và kiểm tra dự đoán trên", len(packaging_predictions), "mẫu.")
